# Pandas入门
在本书的其余部分中，pandas将是一个主要的工具。它包含数据结构和数据操作工具，旨在使Python中的数据清洗和分析变得快速和方便。pandas经常与NumPy和SciPy等数值计算工具、statsmodels和scikit-learn等分析库以及matplotlib等数据可视化库一起使用。pandas采用了NumPy的基于数组计算的惯用风格，特别是基于数组的函数和对无循环数据处理的偏好。

虽然pandas借鉴了许多NumPy的编码风格，但最大的不同是pandas是为处理表格数据或异构数据而设计的。相比之下，NumPy最适合处理同类型的数值数组数据。

自2010年成为开源项目以来，pandas已经发展成为一个相当大的库，适用于广泛的现实世界用例。开发者社区已增长到超过2500名不同的贡献者，他们一直在帮助构建该项目，因为他们使用它来解决日常的数据问题。活跃的pandas开发者和用户社区是其成功的关键部分。

在本书的其余部分，我使用以下NumPy和pandas的导入约定：



In [1]:
import numpy as np
import pandas as pd

因此，在代码中看到pd时，它指的是pandas。你可能会发现将Series和DataFrame导入本地命名空间更容易，因为它们经常被使用：

In [2]:
from pandas import Series, DataFrame

## pandas数据结构介绍

要开始使用pandas，你需要熟悉它的两个主要数据结构：Series和DataFrame。虽然它们不是解决每个问题的通用方案，但它们为各种数据任务提供了一个坚实的基础。

### Series

Series是一维的类似数组的对象，包含一系列值（与NumPy类型相似）和与之相关联的数据标签数组，称为索引。最简单的Series仅由数据数组组成：

In [3]:
obj = pd.Series([4, 7, -5, 3])
obj

0    4
1    7
2   -5
3    3
dtype: int64

交互式显示的Series字符串表示形式在左侧显示索引，右侧显示值。由于我们没有指定数据的索引，因此会创建一个默认的索引，由整数0到N-1组成（其中N是数据的长度）。您可以通过其数组和索引属性分别获取Series的数组表示和索引对象：

In [4]:
obj.array

<NumpyExtensionArray>
[4, 7, -5, 3]
Length: 4, dtype: int64

In [5]:
obj.index

RangeIndex(start=0, stop=4, step=1)

.array属性的结果是PandasArray，它通常包裹一个NumPy数组，但也可以包含特殊的扩展数组类型，这些将在第7.3章“扩展数据类型”中更详细地讨论。

通常情况下，您希望创建一个Series，其索引使用标签标识每个数据点：

In [6]:
obj2 = pd.Series([4, 7, -5, 3], index=['d', 'b', 'a', 'c'])
obj2

d    4
b    7
a   -5
c    3
dtype: int64

In [7]:
obj2.index

Index(['d', 'b', 'a', 'c'], dtype='object')

与NumPy数组相比，在选取单个值或一组值时，可以使用标签作为索引：

In [8]:
obj2["a"]

np.int64(-5)

In [9]:
obj2["d"] = 6
obj2[["c", "a", "d"]]

c    3
a   -5
d    6
dtype: int64

这里["c","a","d"]被解释为一个索引列表，即使它包含的是字符串而不是整数。

使用NumPy函数或类似NumPy的操作（例如用布尔数组过滤、标量乘法或应用数学函数）将保留索引值链接：

In [10]:
obj2[obj2 > 0]

d    6
b    7
c    3
dtype: int64

In [11]:
obj2 * 2

d    12
b    14
a   -10
c     6
dtype: int64

In [12]:
import numpy as np

np.exp(obj2)

d     403.428793
b    1096.633158
a       0.006738
c      20.085537
dtype: float64

将Series视为固定长度、有序的词典是一种思考方式，因为它是一个索引值到数据值的映射。它可以在许多你可能使用字典的上下文中使用：

In [13]:
"b" in obj2

True

In [14]:
"e" in obj2

False

如果你有一个包含数据的Python字典，可以通过传递该字典来创建一个Series：

In [15]:
sdata = {"Ohio": 35000, "Texas": 71000, "Oregon": 16000, "Utah": 5000}

obj3 = pd.Series(sdata)
obj3

Ohio      35000
Texas     71000
Oregon    16000
Utah       5000
dtype: int64

可以使用to_dict方法将一个Series转换回字典：

In [16]:
obj3.to_dict()

{'Ohio': 35000, 'Texas': 71000, 'Oregon': 16000, 'Utah': 5000}

当你只传递一个字典时，结果序列中的索引将根据字典的keys方法尊重键的顺序，这取决于键插入的顺序。你可以通过传递一个索引来覆盖这一点，该索引以你想要它们在结果序列中出现的顺序包含字典键：

In [17]:
states = ["California", "Ohio", "Oregon", "Texas"]
obj4 = pd.Series(sdata, index=states)
obj4

California        NaN
Ohio          35000.0
Oregon        16000.0
Texas         71000.0
dtype: float64

在这里，sdata中找到了三个值并放置在了适当的位置，但由于没有找到“California”的值，它显示为NaN（非数字），这在pandas中被认为是标记缺失或NA值的。由于“Utah”没有被包括在states中，因此它被排除在结果对象之外。

我将“缺失”、“NA”或“空值”这些术语交替使用来指代缺失数据。应该使用pandas中的isna和notna函数来检测缺失数据：

In [18]:
pd.isna(obj4)

California     True
Ohio          False
Oregon        False
Texas         False
dtype: bool

In [19]:
pd.notna(obj4)

California    False
Ohio           True
Oregon         True
Texas          True
dtype: bool

Series还有以下实例方法：

In [20]:
obj4.isna()

California     True
Ohio          False
Oregon        False
Texas         False
dtype: bool

对于许多应用来说，Series 的一个有用特性是它在算术运算中自动按索引标签对齐：

In [21]:
obj3

Ohio      35000
Texas     71000
Oregon    16000
Utah       5000
dtype: int64

In [22]:
obj4

California        NaN
Ohio          35000.0
Oregon        16000.0
Texas         71000.0
dtype: float64

In [23]:
obj3 + obj4

California         NaN
Ohio           70000.0
Oregon         32000.0
Texas         142000.0
Utah               NaN
dtype: float64

数据对齐特性将在后面更详细地讨论。如果你对数据库有经验，你可以将其视为类似于连接操作。

Series对象本身及其索引都有一个name属性，该属性与pandas功能的其他部分集成：

In [24]:
obj4.name = "population"
obj4.index.name = "state"
obj4

state
California        NaN
Ohio          35000.0
Oregon        16000.0
Texas         71000.0
Name: population, dtype: float64

可以通过赋值来改变一个Series的索引：

In [25]:
obj

0    4
1    7
2   -5
3    3
dtype: int64

In [26]:
obj.index = ["Bob", "Steve", "Jeff", "Ryan"]

obj

Bob      4
Steve    7
Jeff    -5
Ryan     3
dtype: int64

### DataFrame

DataFrame表示一个矩形数据表，包含一个有序、命名的列集合，每列可以是不同的值类型（数值型、字符串型、布尔型等）。DataFrame具有行索引和列索引；可以将其视为所有共享相同索引的Series的字典。
> 虽然DataFrame在物理上是二维的，但你可以使用它以表格格式表示更高维度的数据，通过层次索引，这是我们将在第8章讨论的主题：数据整理：连接、组合和重塑，以及pandas中一些更高级数据处理功能的一个组成部分。

构建DataFrame的方法有很多种，尽管最常见的是从等长列表或NumPy数组的字典中创建：

In [27]:
data = {
    "state": ["Ohio", "Ohio", "Ohio", "Nevada", "Nevada", "Nevada"],
    "year": [2000, 2001, 2002, 2001, 2002, 2003],
    "pop": [1.5, 1.7, 3.6, 2.4, 2.9, 3.2],
}

frame = pd.DataFrame(data)

生成的DataFrame将自动分配索引，与Series一样，列的顺序根据数据中键的顺序排列（这取决于它们在字典中的插入顺序）：

In [28]:
frame

,state,year,pop
0,Ohio,2000,1.5
1,Ohio,2001,1.7
2,Ohio,2002,3.6
3,Nevada,2001,2.4
4,Nevada,2002,2.9
5,Nevada,2003,3.2


对于大型数据框，head方法只选择前五行：

In [29]:
frame.head()

,state,year,pop
0,Ohio,2000,1.5
1,Ohio,2001,1.7
2,Ohio,2002,3.6
3,Nevada,2001,2.4
4,Nevada,2002,2.9


同样地，tail 返回最后五行：

In [30]:
frame.tail()

,state,year,pop
1,Ohio,2001,1.7
2,Ohio,2002,3.6
3,Nevada,2001,2.4
4,Nevada,2002,2.9
5,Nevada,2003,3.2


如果您指定了一列序列，DataFrame的列将按照该顺序排列：

In [31]:
pd.DataFrame(data, columns=["year", "state", "pop"])

,year,state,pop
0,2000,Ohio,1.5
1,2001,Ohio,1.7
2,2002,Ohio,3.6
3,2001,Nevada,2.4
4,2002,Nevada,2.9
5,2003,Nevada,3.2


如果传递的列不在字典中，则该列在结果中将显示为空值：

In [32]:
frame2 = pd.DataFrame(data, columns=["year", "state", "pop", "debt"])

frame2

,year,state,pop,debt
0,2000,Ohio,1.5,NaN
1,2001,Ohio,1.7,NaN
2,2002,Ohio,3.6,NaN
3,2001,Nevada,2.4,NaN
4,2002,Nevada,2.9,NaN
5,2003,Nevada,3.2,NaN


In [33]:
frame2.columns

Index(['year', 'state', 'pop', 'debt'], dtype='object')

DataFrame中的列可以通过类似字典的表示法或使用点属性表示法检索为Series：

In [34]:
frame2['state']

0      Ohio
1      Ohio
2      Ohio
3    Nevada
4    Nevada
5    Nevada
Name: state, dtype: object

In [35]:
frame2.year

0    2000
1    2001
2    2002
3    2001
4    2002
5    2003
Name: year, dtype: int64

> 在IPython中，为了方便起见，提供了类似属性的访问（例如，frame2.year）和列名的标签补全。frame2[column]适用于任何列名，但只有当列名是一个有效的Python变量名并且不与DataFrame中的任何方法名冲突时，frame2.column才有效。例如，如果一个列名包含空格或除下划线之外的符号，则无法使用点属性方法进行访问。

请注意，返回的Series具有与DataFrame相同的索引，并且它们的名称属性已适当设置。

也可以通过位置或名称检索行，使用特殊属性iloc和loc（稍后将在DataFrame的loc和iloc选择中详细介绍）：

In [36]:
frame2.loc[1]

year     2001
state    Ohio
pop       1.7
debt      NaN
Name: 1, dtype: object

In [37]:
frame2.iloc[2]

year     2002
state    Ohio
pop       3.6
debt      NaN
Name: 2, dtype: object

列可以通过赋值进行修改。例如，空的债务列可以分配一个标量值或一组值：

In [38]:
frame2['debt'] = 16.5
frame2

,year,state,pop,debt
0,2000,Ohio,1.5,16.5
1,2001,Ohio,1.7,16.5
2,2002,Ohio,3.6,16.5
3,2001,Nevada,2.4,16.5
4,2002,Nevada,2.9,16.5
5,2003,Nevada,3.2,16.5


In [39]:
frame2['debt'] = np.arange(6)
frame2

,year,state,pop,debt
0,2000,Ohio,1.5,0
1,2001,Ohio,1.7,1
2,2002,Ohio,3.6,2
3,2001,Nevada,2.4,3
4,2002,Nevada,2.9,4
5,2003,Nevada,3.2,5


当您将列表或数组分配给列时，值的长度必须与DataFrame的长度相匹配。如果您分配了一个Series，其标签将完全对齐到DataFrame的索引，并在任何不存在的索引值中插入缺失值：

In [40]:
val = pd.Series([-1.2, -1.5, -1.7], index=[2, 4, 5])

frame2['debt'] = val
frame2

,year,state,pop,debt
0,2000,Ohio,1.5,NaN
1,2001,Ohio,1.7,NaN
2,2002,Ohio,3.6,-1.2
3,2001,Nevada,2.4,NaN
4,2002,Nevada,2.9,-1.5
5,2003,Nevada,3.2,-1.7


分配一个不存在的列将创建一个新列。

del 关键字将像使用字典一样删除列。例如，我首先添加一个新列的布尔值，其中状态列等于“Ohio”：

In [41]:
frame2['eastern'] = frame2['state'] == 'Ohio'
frame2

,year,state,pop,debt,eastern
0,2000,Ohio,1.5,NaN,True
1,2001,Ohio,1.7,NaN,True
2,2002,Ohio,3.6,-1.2,True
3,2001,Nevada,2.4,NaN,False
4,2002,Nevada,2.9,-1.5,False
5,2003,Nevada,3.2,-1.7,False


> 无法使用frame2.eastern属性表示法创建新列。

然后可以使用del方法来删除这一列：

In [42]:
del frame2['eastern']
frame2.columns

Index(['year', 'state', 'pop', 'debt'], dtype='object')

> 从DataFrame中索引返回的列是底层数据的视图，而不是副本。因此，对Series的任何原地修改都将在DataFrame中反映出来。可以使用Series的copy方法显式复制该列。

另一种常见的数据形式是嵌套字典的字典：

In [43]:
populations = {
    "Ohio": {2000: 1.5, 2001: 1.7, 2002: 3.6},
    "Nevada": {2001: 2.4, 2002: 2.9},
}
frame3 = pd.DataFrame(populations)
frame3

,Ohio,Nevada
2000,1.5,NaN
2001,1.7,2.4
2002,3.6,2.9


你可以使用与NumPy数组类似的语法来转置DataFrame（交换行和列）：

In [44]:
frame3.T

,2000,2001,2002
Ohio,1.5,1.7,3.6
Nevada,NaN,2.4,2.9


> 请注意，如果列的数据类型不一致，转置会丢弃列数据类型，因此转置后再转置可能会丢失之前的类型信息。在这种情况下，列将变成纯Python对象的数组。

内部字典中的键被组合起来形成结果的索引。如果指定了显式索引，则不是这样：

In [45]:
pd.DataFrame(populations, index=[2001, 2002, 2003])

,Ohio,Nevada
2001,1.7,2.4
2002,3.6,2.9
2003,NaN,NaN


In [46]:
pdata = {"Ohio": frame3["Ohio"][:-1], "Nevada": frame3["Nevada"][:2]}
pd.DataFrame(pdata)

,Ohio,Nevada
2000,1.5,NaN
2001,1.7,2.4


**DataFrame构造函数的可能数据输入**:

| 类型 | 说明 |
|-----|------|
| 二维ndarray | 数据矩阵，可选的行和列标签 |
| 数组、列表或元组的字典 | 每个序列都成为DataFrame中的一列；所有序列的长度必须相同 |
| NumPy结构化/记录数组 | 被视为“数组字典”案例 |
| Series词典 | 每个值成为一个列；如果没有显式索引传递，则将每个Series的索引合并在一起形成结果的行索引。 |
| 词典的词典 | 每个内部字典成为一个列；键被合并以形成行索引，就像“字典的序列”情况一样 |
| 字典列表或Series | 每个项目都成为DataFrame中的一行；字典键或Series索引的并集成为DataFrame的列标签。 |
| 列表或元组的列表 | 被视为“二维ndarray”情况 |
| 另一个DataFrame | 除非传递了不同的索引，否则将使用DataFrame的索引。 |
| NumPy MaskedArray | 类似于“2D ndarray”情况，除了DataFrame结果中缺少被遮蔽的值 |

如果DataFrame的索引和列名属性被设置，这些也会显示出来：

In [47]:
frame3.index.name = "year"
frame3.columns.name = "state"
frame3

state,Ohio,Nevada
year,,
2000,1.5,NaN
2001,1.7,2.4
2002,3.6,2.9


与Series不同，DataFrame没有name属性。DataFrame的to_numpy方法返回DataFrame中包含的数据作为一个二维ndarray：

In [48]:
frame3.to_numpy()

array([[1.5, nan],
       [1.7, 2.4],
       [3.6, 2.9]])

如果Series的列是不同的数据类型，则返回数组的类型将选择以适应所有列：

In [49]:
frame2.to_numpy()

array([[2000, 'Ohio', 1.5, nan],
       [2001, 'Ohio', 1.7, nan],
       [2002, 'Ohio', 3.6, -1.2],
       [2001, 'Nevada', 2.4, nan],
       [2002, 'Nevada', 2.9, -1.5],
       [2003, 'Nevada', 3.2, -1.7]], dtype=object)

### 索引对象

pandas的Index对象负责存储轴标签（包括DataFrame的列名）和其他元数据（如轴名称或名称）。任何您在构建Series或DataFrame时使用的数组或其他标签序列都会内部转换为Index：

In [50]:
obj = pd.Series(np.arange(3), index=["a", "b", "c"])
obj

a    0
b    1
c    2
dtype: int64

In [51]:
index = obj.index
index

Index(['a', 'b', 'c'], dtype='object')

In [52]:
index[1:]

Index(['b', 'c'], dtype='object')

索引对象是不可变的，因此用户不能修改它们：

In [53]:
index[1] = "d"

TypeError: Index does not support mutable operations

不可变性使得在数据结构之间共享索引对象更安全：

In [ ]:
labels = pd.Index(np.arange(3))
labels

Index([0, 1, 2], dtype='int64')

In [ ]:
obj2 = pd.Series([1.5, -2.5, 0], index=labels)
obj2

0    1.5
1   -2.5
2    0.0
dtype: float64

In [ ]:
obj2.index is labels

True

> 有些用户不会经常利用索引提供的能力，但是某些操作会产生包含索引数据的结果，因此了解它们的工作原理是很重要的。

除了像数组一样之外，索引还表现得像一个固定大小的集合：

In [ ]:
frame3

state,Ohio,Nevada
year,,
2000,1.5,NaN
2001,1.7,2.4
2002,3.6,2.9


In [ ]:
frame3.columns

Index(['Ohio', 'Nevada'], dtype='object', name='state')

In [ ]:
"Ohio" in frame3.columns

True

In [ ]:
2003 in frame3.index

False

与Python集合不同，pandas索引可以包含重复的标签：

In [ ]:
pd.Index(["foo", "foo", "bar", "bar"])

Index(['foo', 'foo', 'bar', 'bar'], dtype='object')

选择具有重复标签的选项将选择该标签的所有实例。

**一些索引方法和属性**:

| 方法/属性 | 描述 |
|----------|-----|
| append() | 与额外的索引对象连接，生成一个新的索引 |
| difference() | 计算集合差作为索引 |
| intersection() | 计算集合交集 |
| union() | 计算集合的并集 |
| isin() | 计算布尔数组，指示每个值是否包含在传递的集合中 |
| delete() | 计算删除索引i处的元素后的新索引 |
| drop() | 通过删除传递的值来计算新的索引 |
| insert() | 通过在索引i处插入元素来计算新索引 |
| is_monotonic | 如果每个元素都大于或等于前一个元素，则返回True |
| is_unique | 如果索引没有重复值，则返回True |
| unique() | 计算索引中唯一值的数组 |

## 基本功能

本节将指导您了解与Series或DataFrame中包含的数据进行交互的基本机制。在接下来的章节中，我们将更深入地探讨使用pandas进行数据分析和操作的主题。本书并不打算作为pandas库的详尽文档；相反，我们的重点是让您熟悉常用的功能，而较少使用的（即更晦涩的）内容则留待您在阅读在线pandas文档时进一步学习。

### 重新索引

在pandas对象上的一种重要方法是reindex，这意味着创建一个新对象，其值重新排列以与新索引对齐。考虑一个例子：

In [ ]:
obj = pd.Series([4.5, 7.2, -5.3, 3.6], index=["d", "b", "a", "c"])

obj

d    4.5
b    7.2
a   -5.3
c    3.6
dtype: float64

调用reindex方法会根据新的索引重新排列数据，如果某些索引值之前不存在，则会引入缺失值：

In [ ]:
obj2 = obj.reindex(["a", "b", "c", "d", "e"])
obj2

a   -5.3
b    7.2
c    3.6
d    4.5
e    NaN
dtype: float64

对于有序数据（如时间序列），在重新索引时可能需要进行一些插值或填充。方法选项允许我们这样做，使用诸如ffill的方法向前填充值：

In [ ]:
obj3 = pd.Series(["blue", "purple", "yellow"], index=[1, 2, 4])
obj3

1      blue
2    purple
4    yellow
dtype: object

In [ ]:
obj3.reindex(range(6), method="ffill")

0       NaN
1      blue
2    purple
3    purple
4    yellow
5    yellow
dtype: object

使用DataFrame时，reindex可以更改（行）索引、列或两者。当仅传递一个序列时，它会对结果中的行进行重新索引：

In [ ]:
frame = pd.DataFrame(np.arange(9).reshape((3, 3)),
                     index=["a", "c", "d"],
                     columns=["Ohio", "Texas", "California"])
frame

,Ohio,Texas,California
a,0,1,2
c,3,4,5
d,6,7,8


In [ ]:
frame2 = frame.reindex(["a", "b", "c", "d"])
frame2

,Ohio,Texas,California
a,0.0,1.0,2.0
b,NaN,NaN,NaN
c,3.0,4.0,5.0
d,6.0,7.0,8.0


可以使用columns关键字重新索引列：

In [ ]:
states = ["Texas", "Utah", "California"]
frame.reindex(columns=states)

,Texas,Utah,California
a,1,NaN,2
c,4,NaN,5
d,7,NaN,8


因为“Ohio”不在states中，所以该列的数据从结果中删除。

重新索引特定轴的另一种方法是传递新的轴标签作为位置参数，然后使用axis关键字指定要重新索引的轴：

In [ ]:
frame.reindex(states, axis="columns")

,Texas,Utah,California
a,1,NaN,2
c,4,NaN,5
d,7,NaN,8


**reindex函数参数**:
| 参数 | 描述 |
|------|-----|
| labels | 要使用的新的索引序列。可以是Index实例或任何其他类似序列的Python数据结构。将使用原索引，不进行复制。|
| index | 使用传递的序列作为新的索引标签。|
| columns | 使用传递的序列作为新列标签。 |
| axis | 要重新索引的轴，可以是“索引”（行）或“列”。默认值为“索引”。您也可以交替使用reindex(index=new_labels)或reindex(columns=new_labels)。|
| method | 插值（填充）方法；"ffill"向前填充，而"bfill"向后填充。|
| fill_value | 通过重新索引引入缺失数据时使用的替代值。当您希望结果中的缺失标签具有空值时，请使用fill_value="missing"（默认行为）。|
| limit | 向前填充或向后填充时，要填充的最大间隙（以元素数量计）。|
| tolerance | 向前填充或向后填充时，用于不精确匹配的最大尺寸间隙（以绝对数值距离表示）。|
| level | 在多索引级别上匹配简单索引；否则选择子集。|
| copy | 如果为True，即使新索引与旧索引相同，也始终复制底层数据；如果为False，当索引相同时不复制数据。|

正如我们将在稍后探讨的DataFrame上的loc和iloc选择中看到的，您还可以使用loc操作符进行重索引，许多用户更喜欢始终这样做。这只在所有的新的索引标签都已经在DataFrame中存在时才有效（而reindex将为新标签插入缺失数据）：

In [ ]:
frame

,Ohio,Texas,California
a,0,1,2
c,3,4,5
d,6,7,8


In [ ]:
frame.loc[["a", "d", "c"], ["California", "Texas"]]

,California,Texas
a,2,1
d,8,7
c,5,4


### 从轴中删除条目

如果你已经有一个索引数组或列表，并且没有那些条目，那么从轴上删除一个或多个条目很简单，因为你可以使用reindex方法或基于.loc的索引。由于这可能需要一些数据清洗和集合逻辑，drop方法将返回一个新对象，其中从轴上删除了指定的值或值：

In [ ]:
obj = pd.Series(np.arange(5), index=["a", "b", "c", "d", "e"])
obj

a    0
b    1
c    2
d    3
e    4
dtype: int64

In [ ]:
new_obj = obj.drop("c")
new_obj

a    0
b    1
d    3
e    4
dtype: int64

In [ ]:
obj.drop(["d", "c"])

a    0
b    1
e    4
dtype: int64

使用DataFrame，可以从任一轴删除索引值。为了说明这一点，我们首先创建一个示例DataFrame：

In [ ]:
data = pd.DataFrame(np.arange(16).reshape((4, 4)),
                    index=["Ohio", "Colorado", "Utah", "New York"],
                    columns=["one", "two", "three", "four"])
data

,one,two,three,four
Ohio,0,1,2,3
Colorado,4,5,6,7
Utah,8,9,10,11
New York,12,13,14,15


使用一系列标签调用drop将删除行标签（轴0）中的值：

In [ ]:
data.drop(index=["Colorado", "Ohio"])

,one,two,three,four
Utah,8,9,10,11
New York,12,13,14,15


要从列中删除标签，请改用columns关键字：

In [ ]:
data.drop(columns=["two"])

,one,three,four
Ohio,0,2,3
Colorado,4,6,7
Utah,8,10,11
New York,12,14,15


您还可以通过传递axis=1（类似于NumPy）或axis="columns"来从列中删除值：

In [ ]:
data.drop("two", axis=1)

,one,three,four
Ohio,0,2,3
Colorado,4,6,7
Utah,8,10,11
New York,12,14,15


In [ ]:
data.drop(["two", "four"], axis="columns")

,one,three
Ohio,0,2
Colorado,4,6
Utah,8,10
New York,12,14


### 索引、选择和过滤

Series索引（obj[...]）的工作方式类似于NumPy数组索引，不同之处在于你可以使用系列的索引值而不仅仅是整数。这里有一些例子：

In [ ]:
obj = pd.Series(np.arange(4), index=["a", "b", "c", "d"])
obj

a    0
b    1
c    2
d    3
dtype: int64

In [ ]:
obj['b']

np.int64(1)

In [ ]:
obj[1]

/var/folders/c5/vh03t8zn4797kc18lgrjtcbr0000gn/T/ipykernel_20676/2469632899.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  obj[1]


np.int64(1)

In [ ]:
obj[2:4]

c    2
d    3
dtype: int64

In [ ]:
obj[["b", "a", "d"]]

b    1
a    0
d    3
dtype: int64

In [ ]:
obj[[1, 3]]

/var/folders/c5/vh03t8zn4797kc18lgrjtcbr0000gn/T/ipykernel_20676/2982346117.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  obj[[1, 3]]


b    1
d    3
dtype: int64

In [ ]:
obj[obj<2]

a    0
b    1
dtype: int64

虽然可以通过标签选择数据，但首选的索引值选择方法是使用特殊的loc操作符：

In [ ]:
obj.loc[["b", "a", "d"]]


b    1
a    0
d    3
dtype: int64

之所以更倾向于使用loc是因为在索引时对整数有不同的处理方式。常规的基于[]的索引会将包含整数的索引视为标签，因此其行为会根据索引的数据类型而有所不同。例如：

In [ ]:
obj1 = pd.Series([1, 2, 3], index=[2, 0, 1])
obj1

2    1
0    2
1    3
dtype: int64

In [ ]:
obj1[[0, 1, 2]]

0    2
1    3
2    1
dtype: int64

In [ ]:
obj2 = pd.Series([1, 2, 3], index=["a", "b", "c"])
obj2

a    1
b    2
c    3
dtype: int64

In [ ]:
obj2[[0, 1, 2]]

/var/folders/c5/vh03t8zn4797kc18lgrjtcbr0000gn/T/ipykernel_20676/2599987575.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  obj2[[0, 1, 2]]


a    1
b    2
c    3
dtype: int64

使用loc时，表达式obj.loc[[0, 1, 2]]在索引不包含整数时会失败：

In [ ]:
obj2.loc[[0, 1]]

KeyError: "None of [Index([0, 1], dtype='int64')] are in the [index]"

由于loc操作符仅使用标签索引，因此还有一个iloc操作符专门使用整数索引，无论索引是否包含整数都能保持一致的工作方式：

In [ ]:
obj1.iloc[[0, 1, 2]]

2    1
0    2
1    3
dtype: int64

In [ ]:
obj2.iloc[[0, 1, 2]]

a    1
b    2
c    3
dtype: int64

**⚠️你也可以使用标签切片，但它的行为与普通的Python切片不同，因为终点是包含的：**

In [ ]:
obj2.loc["b":"c"]

b    2
c    3
dtype: int64

使用这些方法分配值会修改序列的相应部分：

In [ ]:
obj2.loc["b":"c"] = 5
obj2

a    1
b    5
c    5
dtype: int64

> 尝试使用loc或iloc之类的函数而不是用方括号“索引到”它们是一个常见的新手错误。方括号表示法用于启用切片操作，并允许在DataFrame对象上对多个轴进行索引。

索引到DataFrame中检索一个或多个列，可以是一个单一值或序列：

In [ ]:
data = pd.DataFrame(np.arange(16).reshape((4, 4)),
                    index=["Ohio", "Colorado", "Utah", "New York"],
                    columns=["one", "two", "three", "four"])
data

,one,two,three,four
Ohio,0,1,2,3
Colorado,4,5,6,7
Utah,8,9,10,11
New York,12,13,14,15


In [ ]:
data['two']

Ohio         1
Colorado     5
Utah         9
New York    13
Name: two, dtype: int64

In [ ]:
data[['two']]

,two
Ohio,1
Colorado,5
Utah,9
New York,13


In [ ]:
data[['three', 'one']]

,three,one
Ohio,2,0
Colorado,6,4
Utah,10,8
New York,14,12


这种索引有一些特殊情况。第一个是使用布尔数组切片或选择数据：

In [ ]:
data[:2]

,one,two,three,four
Ohio,0,1,2,3
Colorado,4,5,6,7


In [ ]:
data[data['three'] > 5]

,one,two,three,four
Colorado,4,5,6,7
Utah,8,9,10,11
New York,12,13,14,15


行选择语法数据[:2]是为了方便提供的。将单个元素或列表传递给[]操作符来选择列。

另一个用例是使用布尔值数据框进行索引，例如通过标量比较生成的数据框。考虑一个数据框，其中所有布尔值都是通过与标量值比较产生的：

In [ ]:
data < 5

,one,two,three,four
Ohio,True,True,True,True
Colorado,True,False,False,False
Utah,False,False,False,False
New York,False,False,False,False


我们可以使用这个DataFrame来将值为True的每个位置的值赋为0，如下所示：

In [ ]:
data[data<5] = 0
data

,one,two,three,four
Ohio,0,0,0,0
Colorado,0,5,6,7
Utah,8,9,10,11
New York,12,13,14,15


#### 使用loc和iloc对DataFrame进行选择

与Series类似，DataFrame具有基于标签和整数的索引的特殊属性loc和iloc。由于DataFrame是二维的，您可以使用类似于NumPy的符号通过轴标签（loc）或整数（iloc）选择行和列的子集。

作为第一个示例，让我们通过标签选择单行：

In [ ]:
data

,one,two,three,four
Ohio,0,0,0,0
Colorado,0,5,6,7
Utah,8,9,10,11
New York,12,13,14,15


In [ ]:
data.loc["Colorado"]

one      0
two      5
three    6
four     7
Name: Colorado, dtype: int64

选择单行会返回一个Series对象，其索引包含DataFrame的列标签。要选择多个角色，创建一个新的DataFrame，通过传递一系列标签：

In [ ]:
data.loc[["Colorado", "New York"]]

,one,two,three,four
Colorado,0,5,6,7
New York,12,13,14,15


在loc中，可以通过逗号分隔行和列选择来组合它们：

In [ ]:
data.loc["Colorado", ["two", "three"]]

two      5
three    6
Name: Colorado, dtype: int64

然后我们将使用iloc对整数执行一些类似的筛选：

In [ ]:
data.iloc[2]

one       8
two       9
three    10
four     11
Name: Utah, dtype: int64

In [ ]:
data.iloc[[2, 1]]

,one,two,three,four
Utah,8,9,10,11
Colorado,0,5,6,7


In [ ]:
data.iloc[2, [3, 0, 1]]

four    11
one      8
two      9
Name: Utah, dtype: int64

In [ ]:
data.iloc[[1, 2], [3, 0, 1]]

,four,one,two
Colorado,7,0,5
Utah,11,8,9


索引函数除了单个标签或标签列表外，还可以处理切片：

In [ ]:
data.loc[:"Utah", "two"]

Ohio        0
Colorado    5
Utah        9
Name: two, dtype: int64

In [ ]:
data.iloc[:, :3][data.three > 5]

,one,two,three
Colorado,0,5,6
Utah,8,9,10
New York,12,13,14


布尔数组可以与loc一起使用，但不能与iloc一起使用：

In [ ]:
data.loc[data.three >= 2]

,one,two,three,four
Colorado,0,5,6,7
Utah,8,9,10,11
New York,12,13,14,15


有许多方法可以选择和重新排列包含在pandas对象中的数据。对于DataFrame，下表提供了其中许多方法的简要概述。如后所述，还有多种额外的选项可用于处理层次索引。

| 类型 | 说明 |
|-----|------|
| df[column] | 从DataFrame中选择单列或列序列；特殊情况便利性：布尔数组（过滤行）、切片（切片行）或布尔DataFrame（基于某些标准设置值）|
| df.loc[rows] | 通过标签从DataFrame中选择单行或行子集 |
| df.loc[:, cols] | 通过标签选择单列或列子集 |
| df.loc[rows, cols] | 通过标签选择行和列 |
| df.iloc[rows] | 通过整数位置从DataFrame中选择单行或行子集 |
| df.iloc[:, cols] | 通过整数位置选择单列或列子集 |
| df.iloc[rows, cols] | 通过整数位置选择行和列 |
| df.at[row, col] | 通过行和列标签选择一个单一的标量值 |
| df.iat[row, col] | 按行和列位置（整数）选择一个单一的标量值 |
| reindex | 通过标签选择行或列 |

#### 整数索引陷阱

使用整数索引的pandas对象对于新用户来说可能是一个绊脚石，因为它们与内置的Python数据结构（如列表和元组）的工作方式不同。例如，您可能不会期望以下代码产生错误：

In [ ]:
ser = pd.Series(np.arange(3.0))
ser

0    0.0
1    1.0
2    2.0
dtype: float64

In [ ]:
ser[-1]

KeyError: -1

在这种情况下，pandas可以“回退”到整数索引，但通常很难做到这一点而不在用户代码中引入微妙的错误。这里我们有一个包含0、1和2的索引，但pandas不想猜测用户想要什么（基于标签的索引或基于位置的）.

另一方面，对于非整数索引，则不存在这种歧义：

In [ ]:
ser2 = pd.Series(np.arange(3.0), index=['a', 'b', 'c'])
ser2[-1]

/var/folders/c5/vh03t8zn4797kc18lgrjtcbr0000gn/T/ipykernel_20676/2408619474.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  ser2[-1]


np.float64(2.0)

如果您有一个包含整数的轴索引，数据选择将始终面向标签。如上所述，如果您使用loc（针对标签）或iloc（针对整数），您将得到您想要的精确内容：

In [ ]:
ser.iloc[-1]

np.float64(2.0)

另一方面，用整数切片总是以整数为导向的：

In [ ]:
ser[:2]

0    0.0
1    1.0
dtype: float64

为了避免歧义，最好总是使用loc和iloc进行索引。

#### 链式索引的陷阱

在前一节中，我们讨论了如何使用loc和iloc在DataFrame上进行灵活的选择。这些索引属性也可以用来就地修改DataFrame对象，但这样做需要一些小心。

例如，在上面的示例DataFrame中，我们可以通过标签或整数位置来分配给列或行：

In [ ]:
data.loc[:, "one"] = 1
data

,one,two,three,four
Ohio,1,0,0,0
Colorado,1,5,6,7
Utah,1,9,10,11
New York,1,13,14,15


In [ ]:
data.iloc[2] = 5
data

,one,two,three,four
Ohio,1,0,0,0
Colorado,1,5,6,7
Utah,5,5,5,5
New York,1,13,14,15


In [ ]:
data.loc[data["four"] > 5] = 3
data

,one,two,three,four
Ohio,1,0,0,0
Colorado,3,3,3,3
Utah,5,5,5,5
New York,3,3,3,3


新手经常犯的一个错误是在赋值时链式选择，例如：

In [ ]:
data.loc[data.three == 5]["three"] = 6

/var/folders/c5/vh03t8zn4797kc18lgrjtcbr0000gn/T/ipykernel_20676/867481848.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data.loc[data.three == 5]["three"] = 6


根据数据内容，这可能会打印一个特殊的SettingWithCopyWarning警告，它警告你正在尝试修改一个临时值（data.loc[data.three == 5]的非空结果），而不是原始DataFrame数据，这可能是你原本打算的。这里，data没有被修改：

In [ ]:
data

,one,two,three,four
Ohio,1,0,0,0
Colorado,3,3,3,3
Utah,5,5,5,5
New York,3,3,3,3


在这些情况下，修复方法是重写链式赋值语句以使用单个loc操作符：

In [ ]:
data.loc[data.three == 5, "three"] = 6
data

,one,two,three,four
Ohio,1,0,0,0
Colorado,3,3,3,3
Utah,5,5,6,5
New York,3,3,3,3


在进行赋值时，一个好的经验法则是避免链式索引。在pandas中还有其他会产生SettingWithCopyWarning的情况与链式索引有关。我推荐您在线pandas文档中查看这个话题。

### 算术和数据对齐

pandas可以使处理具有不同索引的对象变得更加简单。例如，当你添加对象时，如果任何索引对不相同，结果中的相应索引将是这些索引对的并集。让我们看一个例子：

In [ ]:
s1 = pd.Series([7.3, -2.5, 3.4, 1.5], index=['a', 'c', 'd', 'e'])
s1

a    7.3
c   -2.5
d    3.4
e    1.5
dtype: float64

In [ ]:
s2 = pd.Series([-2.1, 3.6, -1.5, 4.0], index=['a', 'c', 'e', 'f'])
s2

a   -2.1
c    3.6
e   -1.5
f    4.0
dtype: float64

将这些加起来得到：

In [ ]:
s1 + s2

a    5.2
c    1.1
d    NaN
e    0.0
f    NaN
dtype: float64

内部数据对齐会在不重叠的标签位置引入缺失值。然后这些缺失值会进一步在算术计算中传播。

在DataFrame的情况下，对齐是在行和列上进行的：

In [ ]:
df1 = pd.DataFrame(np.arange(9.).reshape((3, 3)),
                   columns=list('bcd'),
                   index=['Ohio', 'Texas', 'Colorado'])
df1

,b,c,d
Ohio,0.0,1.0,2.0
Texas,3.0,4.0,5.0
Colorado,6.0,7.0,8.0


In [ ]:
df2 = pd.DataFrame(np.arange(12.).reshape((4, 3)),
                   columns=list('bde'),
                   index=['Utah', 'Ohio', 'Texas', 'Oregon'])
df2

,b,d,e
Utah,0.0,1.0,2.0
Ohio,3.0,4.0,5.0
Texas,6.0,7.0,8.0
Oregon,9.0,10.0,11.0


添加这些返回一个DataFrame，其索引和列是每个DataFrame中索引和列的并集：

In [ ]:
df1 + df2

,b,c,d,e
Colorado,NaN,NaN,NaN,NaN
Ohio,3.0,NaN,6.0,NaN
Oregon,NaN,NaN,NaN,NaN
Texas,9.0,NaN,12.0,NaN
Utah,NaN,NaN,NaN,NaN


由于“c”和“e”列在两个DataFrame对象中没有都找到，因此在结果中它们显示为缺失。对于标签在两个对象中都不共有的行也是如此。

如果添加的DataFrame对象没有共同的列名或行标签，那么结果将包含所有空值：

In [ ]:
df1 = pd.DataFrame({"A": [1, 2]})
df1

,A
0,1
1,2


In [ ]:
df2 = pd.DataFrame({"B": [3, 4]})
df2

,B
0,3
1,4


In [ ]:
df1 + df2

,A,B
0,NaN,NaN
1,NaN,NaN


#### 带填充值的算术方法

在具有不同索引的对象之间的算术运算中，当在一个对象中找到轴标签但在另一个对象中没有找到时，您可能希望用特殊值（如0）填充。以下是一个示例，我们通过将其赋值为np.nan来设置特定值为NA（空）：

In [ ]:
df1 = pd.DataFrame(np.arange(12).reshape((3, 4)),
                   columns=list('abcd'))
df1

,a,b,c,d
0,0,1,2,3
1,4,5,6,7
2,8,9,10,11


In [ ]:
df2 = pd.DataFrame(np.arange(20).reshape((4, 5)),
                   columns=list('abcde'))
df2

,a,b,c,d,e
0,0,1,2,3,4
1,5,6,7,8,9
2,10,11,12,13,14
3,15,16,17,18,19


In [ ]:
df1 + df2

,a,b,c,d,e
0,0.0,2.0,4.0,6.0,NaN
1,9.0,11.0,13.0,15.0,NaN
2,18.0,20.0,22.0,24.0,NaN
3,NaN,NaN,NaN,NaN,NaN


使用df1上的add方法，我传递了df2和fill_value参数，该参数在操作中替换任何缺失值：

In [ ]:
df1.add(df2, fill_value=0)

,a,b,c,d,e
0,0.0,2.0,4.0,6.0,4.0
1,9.0,11.0,13.0,15.0,9.0
2,18.0,20.0,22.0,24.0,14.0
3,15.0,16.0,17.0,18.0,19.0


请参见下表，了解用于算术的 Series 和 DataFrame 方法列表。每个方法都有一个以字母 r 开头的对应方法，其参数顺序相反。因此，以下两个语句是等效的：

In [ ]:
1 / df1

,a,b,c,d
0,inf,1.000000,0.500000,0.333333
1,0.250,0.200000,0.166667,0.142857
2,0.125,0.111111,0.100000,0.090909


In [ ]:
df1.rdiv(1)

,a,b,c,d
0,inf,1.000000,0.500000,0.333333
1,0.250,0.200000,0.166667,0.142857
2,0.125,0.111111,0.100000,0.090909


相关地，在重新索引Series或DataFrame时，您还可以指定不同的填充值：

In [ ]:
df1.reindex(columns=df2.columns, fill_value=0)

,a,b,c,d,e
0,0,1,2,3,0
1,4,5,6,7,0
2,8,9,10,11,0


**算术方法**:

| 方法 | 描述 |
|-----|------|
| add, radd | 加法（+）的方法 |
| sub, rsub | 减法方法 |
| div, rdiv | 除法（/）的方法 |
| floordiv, rfloordiv | 地板除法（//）的方法 |
| mul, rmul | 乘法（*）的方法 |
| pow, rpow | 指数运算方法（**） |

#### DataFrame与Series之间的操作

与不同维度的NumPy数组一样，DataFrame和Series之间的算术运算也是定义好的。首先，作为一个激励示例，考虑一个二维数组及其一行之间的差异：

In [ ]:
arr = np.arange(12).reshape((3, 4))

arr

array([[ 0,  1,  2,  3],
       [ 4,  5,  6,  7],
       [ 8,  9, 10, 11]])

In [ ]:
arr[0]

array([0, 1, 2, 3])

In [ ]:
arr - arr[0]

array([[0, 0, 0, 0],
       [4, 4, 4, 4],
       [8, 8, 8, 8]])

当我们从arr中减去arr[0]时，对每一行执行一次减法。这被称为广播，在附录A：高级NumPy中更详细地解释了它与一般NumPy数组的关系。DataFrame和Series之间的操作是相似的：

In [ ]:
frame = pd.DataFrame(np.arange(12).reshape((4, 3)),
                     columns=list('bde'),
                     index=['Utah', 'Ohio', 'Texas', 'Oregon'])
frame

,b,d,e
Utah,0,1,2
Ohio,3,4,5
Texas,6,7,8
Oregon,9,10,11


In [ ]:
series = frame.iloc[0]
series

b    0
d    1
e    2
Name: Utah, dtype: int64

默认情况下，DataFrame和Series之间的算术运算会根据Series的索引在DataFrame的列上进行匹配，然后向下广播到行上：

In [ ]:
frame - series

,b,d,e
Utah,0,0,0
Ohio,3,3,3
Texas,6,6,6
Oregon,9,9,9


如果在DataFrame的列或Series的索引中找不到索引值，则将对象重新索引以形成联合：

In [ ]:
series2 = pd.Series(range(3), index=['b', 'e', 'f'])
series2

b    0
e    1
f    2
dtype: int64

In [ ]:
frame + series2

,b,d,e,f
Utah,0.0,NaN,3.0,NaN
Ohio,3.0,NaN,6.0,NaN
Texas,6.0,NaN,9.0,NaN
Oregon,9.0,NaN,12.0,NaN


如果您想按列广播并在行上匹配，您必须使用一种算术方法并指定在索引上进行匹配。例如：

In [ ]:
series3 = frame['d']
series3

Utah       1
Ohio       4
Texas      7
Oregon    10
Name: d, dtype: int64

In [ ]:
frame

,b,d,e
Utah,0,1,2
Ohio,3,4,5
Texas,6,7,8
Oregon,9,10,11


In [ ]:
frame.sub(series3, axis='index')

,b,d,e
Utah,-1,0,1
Ohio,-1,0,1
Texas,-1,0,1
Oregon,-1,0,1


您传递的轴是要匹配的轴。在这种情况下，我们打算根据DataFrame的行索引（轴="index"）进行匹配，并在列中广播。

### 函数应用与映射

NumPy ufuncs（元素级数组方法）也适用于pandas对象：

In [ ]:
frame = pd.DataFrame(np.random.standard_normal((4, 3)),
                     columns=list('bde'),
                     index=['Utah', 'Ohio', 'Texas', 'Oregon'])
frame

,b,d,e
Utah,0.484318,-1.376674,0.796522
Ohio,0.551370,-1.423772,-0.211787
Texas,1.299594,0.290964,-0.425859
Oregon,1.442736,-0.776689,0.842972


In [ ]:
np.abs(frame)

,b,d,e
Utah,0.484318,1.376674,0.796522
Ohio,0.551370,1.423772,0.211787
Texas,1.299594,0.290964,0.425859
Oregon,1.442736,0.776689,0.842972


另一个常见的操作是对一维数组中的每一列或行应用一个函数。DataFrame的apply方法正是这样做的：

In [ ]:
def f(x):
    return x.max() - x.min()

frame.apply(f)

b    0.958419
d    1.714735
e    1.268831
dtype: float64

这里函数f计算了frame中每列的最大值和最小值之间的差异。结果是一个以frame的列为索引的Series。

如果您传递axis="columns"来应用，函数将改为每行调用一次。思考这个问题的有益方式是“跨列应用”：

In [ ]:
frame.apply(f, axis='columns')

Utah      2.173196
Ohio      1.975141
Texas     1.725452
Oregon    2.219426
dtype: float64

许多最常见的数组统计（如总和和平均值）是DataFrame方法，因此使用apply是不必要的。

传递给apply的函数不需要返回一个标量值；它也可以返回一个包含多个值的Series：

In [ ]:
def f2(x):
    return pd.Series([x.min(), x.max()], index=['min', 'max'])

frame.apply(f2)

,b,d,e
min,0.484318,-1.423772,-0.425859
max,1.442736,0.290964,0.842972


也可以使用元素级Python函数。假设你想从frame中的每个浮点值计算一个格式化的字符串。你可以使用applymap来实现这一点：

In [ ]:
def my_format(x):
    return f'{x:.2f}'

frame.applymap(my_format)

/var/folders/c5/vh03t8zn4797kc18lgrjtcbr0000gn/T/ipykernel_32318/4113676589.py:4: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  frame.applymap(my_format)


,b,d,e
Utah,0.48,-1.38,0.80
Ohio,0.55,-1.42,-0.21
Texas,1.30,0.29,-0.43
Oregon,1.44,-0.78,0.84


applymap这个名字的由来是因为Series有一个map方法用于应用一个元素级别的函数：

In [ ]:
frame['e'].map(my_format)

Utah       0.80
Ohio      -0.21
Texas     -0.43
Oregon     0.84
Name: e, dtype: object

### 排序和排名

按某个标准对数据集进行排序是另一个重要的内置操作。要按行或列标签进行字典顺序排序，请使用sort_index方法，该方法返回一个新的已排序对象：

In [54]:
obj = pd.Series(range(4), index=['d', 'a', 'b', 'c'])
obj

d    0
a    1
b    2
c    3
dtype: int64

In [55]:
obj.sort_index()

a    1
b    2
c    3
d    0
dtype: int64

使用DataFrame，您可以通过任一轴对索引进行排序：

In [56]:
frame = pd.DataFrame(np.arange(8).reshape((2, 4)),
                     index=['three', 'one'],
                     columns=['d', 'a', 'b', 'c'])
frame

,d,a,b,c
three,0,1,2,3
one,4,5,6,7


In [57]:
frame.sort_index()

,d,a,b,c
one,4,5,6,7
three,0,1,2,3


In [58]:
frame.sort_index(axis="columns")

,a,b,c,d
three,1,2,3,0
one,5,6,7,4


数据默认按升序排列，但也可以按降序排列：

In [59]:
frame.sort_index(axis="columns", ascending=False)

,d,c,b,a
three,0,3,2,1
one,4,7,6,5


要按值对Series进行排序，请使用其sort_values方法：

In [60]:
obj = pd.Series([4, 7, -3, 2])
obj.sort_values()

2   -3
3    2
0    4
1    7
dtype: int64

任何缺失值默认都会被排序到序列的末尾：

In [61]:
obj = pd.Series([4, np.nan, 7, np.nan, -3, 2])
obj.sort_values()

4   -3.0
5    2.0
0    4.0
2    7.0
1    NaN
3    NaN
dtype: float64

In [62]:
obj.sort_values(ascending=False)

2    7.0
0    4.0
5    2.0
4   -3.0
1    NaN
3    NaN
dtype: float64

缺失值可以通过使用na_position选项来排序到开头：

In [63]:
obj.sort_values(na_position='first')

1    NaN
3    NaN
4   -3.0
5    2.0
0    4.0
2    7.0
dtype: float64

在排序DataFrame时，可以使用一个或多个列中的数据作为排序键。为此，将一个或多个列名传递给sort_values：

In [64]:
frame = pd.DataFrame({"b": [4, 7, -3, 2], "a": [0, 1, 0, 1]})
frame

,b,a
0,4,0
1,7,1
2,-3,0
3,2,1


In [65]:
frame.sort_values("b")

,b,a
2,-3,0
3,2,1
0,4,0
1,7,1


要按多个列排序，请传递一个名称列表：

In [66]:
frame.sort_values(["a", "b"])

,b,a
2,-3,0
0,4,0
3,2,1
1,7,1


排名从1到数组中有效数据点的数量进行分配，从最低值开始。Series和DataFrame的rank方法值得查看；默认情况下，rank通过为每个组分配平均排名来打破平局：

In [67]:
obj = pd.Series([7, -5, 7, 4, 2, 0, 4])

obj.rank()

0    6.5
1    1.0
2    6.5
3    4.5
4    3.0
5    2.0
6    4.5
dtype: float64

也可以根据在数据中观察到的顺序来分配等级：

In [68]:
obj.rank(method='first')

0    6.0
1    1.0
2    7.0
3    4.0
4    3.0
5    2.0
6    5.0
dtype: float64

在这里，对于条目0和2，没有使用平均排名6.5，而是分别设置为6和7，因为在数据中标签0在标签2之前。

你也可以按降序排列：

In [69]:
obj.rank(ascending=False)

0    1.5
1    7.0
2    1.5
3    3.5
4    5.0
5    6.0
6    3.5
dtype: float64

DataFrame可以计算行或列的排名：

In [71]:
frame = pd.DataFrame({"b": [4.3, 7, -3, 2], "a": [0, 1, 0, 1], "c": [-2, 5, 8, -2.5]})

frame

,b,a,c
0,4.3,0,-2.0
1,7.0,1,5.0
2,-3.0,0,8.0
3,2.0,1,-2.5


In [72]:
frame.rank(axis='columns')

,b,a,c
0,3.0,2.0,1.0
1,3.0,1.0,2.0
2,1.0,2.0,3.0
3,3.0,2.0,1.0


**并列策略：**
| 方法 | 描述 |
|-----|------|
| average | 默认值：将平均等级分配给等组中的每个条目 |
| min | 并列组取该组最小名次 |
| max | 并列组取该组最大名次 |
| first | 按原出现顺序分配名次 |
| dense | 类似于method="min"，但是组与组之间的排名总是增加1而不是组内相等元素的数量 |

### 具有重复标签的轴索引

到目前为止，我们几乎看到的所有示例都有唯一的轴标签（索引值）。虽然许多pandas函数（如reindex）要求标签是唯一的，但这不是强制性的。让我们考虑一个具有重复索引的小Series：

In [77]:
obj = pd.Series(np.arange(5), index=['a', 'a', 'b', 'b', 'c'])

obj

a    0
a    1
b    2
b    3
c    4
dtype: int64

索引的is_unique属性可以告诉你其标签是否唯一：

In [74]:
obj.index.is_unique

False

数据选择是处理重复项时表现不同的地方之一。对包含多个条目的标签进行索引会返回一个Series对象，而对单个条目进行索引则会返回一个标量值：

In [75]:
obj["a"]

a    0
a    1
dtype: int64

In [78]:
obj["c"]

np.int64(4)

这可能会使代码更加复杂，因为索引的输出类型可能根据标签是否重复而变化。

同样的逻辑也适用于索引DataFrame中的行（或列）：

In [79]:
df = pd.DataFrame(np.random.standard_normal((5, 3)),
                  index=['a', 'a', 'b', 'b', 'c'])
df

,0,1,2
a,-0.114896,-0.254808,-0.928318
a,-0.273802,-1.298578,-0.965211
b,0.179092,-0.861642,0.921262
b,0.085966,0.308487,-0.792601
c,0.315861,0.447371,1.194661


In [80]:
df.loc['b']

,0,1,2
b,0.179092,-0.861642,0.921262
b,0.085966,0.308487,-0.792601


In [81]:
df.loc['c']

0    0.315861
1    0.447371
2    1.194661
Name: c, dtype: float64

## 总结并计算描述性统计数据

pandas对象配备了一组常见的数学和统计方法。这些方法大多属于降维或摘要统计类别，它们从Series中提取单个值（如总和或平均值），或者从DataFrame的行或列中提取一系列值。与NumPy数组上发现的类似方法相比，它们内置了对缺失数据的支持。考虑一个小型DataFrame：

In [82]:
df = pd.DataFrame([[1.4, np.nan], [7.1, -4.5], [np.nan, np.nan], [0.75, -1.3]],
                  index=['a', 'b', 'c', 'd'],
                  columns=['one', 'two'])
df

,one,two
a,1.40,NaN
b,7.10,-4.5
c,NaN,NaN
d,0.75,-1.3


调用DataFrame的sum方法返回一个包含列总和的Series：

In [86]:
df.sum()

one    9.25
two   -5.80
dtype: float64

通过axis="columns"或axis=1将跨列求和：

In [87]:
df.sum(axis="columns")

a    1.40
b    2.60
c    0.00
d   -0.55
dtype: float64

如果整行或整列都是NA值，则求和结果为0；如果有任何非NA值，则结果为NA。这可以通过skipna选项禁用，在这种情况下，行或列中的任何NA值都会使相应的结果成为NA：

In [88]:
df.sum(axis="index", skipna=False)

one   NaN
two   NaN
dtype: float64

In [89]:
df.sum(axis="columns", skipna=False)

a     NaN
b    2.60
c     NaN
d   -0.55
dtype: float64

某些聚合操作（如平均值）至少需要一个非空值才能产生一个数值结果，因此我们有：

In [90]:
df.mean(axis="columns")

a    1.400
b    1.300
c      NaN
d   -0.275
dtype: float64

**聚合方法的选项：**
| 选项 | 描述 |
|------|-----|
| axis | 要聚合的轴；对于DataFrame的行是“index”，对于列是“columns” |
| skipna | 是否排除缺失值；默认值为真 |
| level | 如果轴是层次索引（多索引），则按级别分组。|

有些方法，如idxmin和idxmax，返回间接统计数据，如最小值或最大值所在的索引值：

In [92]:
df.idxmax()

one    b
two    d
dtype: object

其他方法是积累：

In [93]:
df.cumsum()

,one,two
a,1.40,NaN
b,8.50,-4.5
c,NaN,NaN
d,9.25,-5.8


有些方法既不是归纳也不是演绎。描述统计就是一个例子，它一次性产生多个摘要统计数据：

In [94]:
df.describe()

,one,two
count,3.000000,2.000000
mean,3.083333,-2.900000
std,3.493685,2.262742
min,0.750000,-4.500000
25%,1.075000,-3.700000
50%,1.400000,-2.900000
75%,4.250000,-2.100000
max,7.100000,-1.300000


在非数值数据上，描述产生替代的摘要统计数据：

In [95]:
obj = pd.Series(["a", "a", "b", "c"] * 4)

obj.describe()

count     16
unique     3
top        a
freq       8
dtype: object

**描述性统计和总结性统计方法**

| 方法 | 描述 |
|-----|------|
| count | 非NA值的数量 |
| describe | 计算摘要统计数据集 |
| min, max | 计算最小值和最大值 |
| argmin, argmax | 计算分别获得最小值或最大值的索引位置（整数）；在DataFrame对象上不可用 |
| idxmin, idxmax | 计算分别获得最小值或最大值的索引标签 |
| quantile | 计算从0到1（默认值：0.5）的样本分位数 |
| sum | 数值之和 |
| mean | 平均值 |
| median | 数值的算术中位数（50%分位数） |
| mad | 平均值绝对偏差 |
| prod | 所有值的乘积 |
| var | 值的样本方差 |
| std | 值的样本标准差 |
| skew | 值的样本偏度（三阶矩） |
| kurt | 值的样本峰度（第四矩） |
| cumsum | 值的累积和 |
| cummin, cummax | 值的累积最小值或最大值 |
| cumprod | 值的累积乘积 |
| diff | 计算第一个算术差分（对时间序列很有用）|
| pct_change | 计算百分比变化 |

### 相关性和协方差

一些摘要统计量，如相关性和协方差，是从成对的参数中计算得出的。让我们考虑一些原本从雅虎财经获取的股价和成交量数据框，这些数据框可以在书的附带数据集中找到的二进制Python pickle文件中获取：

In [97]:
price = pd.read_pickle("examples/yahoo_price.pkl")
volume = pd.read_pickle("examples/yahoo_volume.pkl")

我现在计算价格的百分比变化，这是一个将在第11章时间序列：进一步探讨的时间序列操作。

In [98]:
returns = price.pct_change()
returns.tail()

,AAPL,GOOG,IBM,MSFT
Date,,,,
2016-10-17,-0.000680,0.001837,0.002072,-0.003483
2016-10-18,-0.000681,0.019616,-0.026168,0.007690
2016-10-19,-0.002979,0.007846,0.003583,-0.002255
2016-10-20,-0.000512,-0.005652,0.001719,-0.004867
2016-10-21,-0.003930,0.003011,-0.012474,0.042096


corr方法计算两个Series中重叠的、非缺失值的对齐按索引值的相关性。相关地，cov计算协方差：

In [99]:
returns["MSFT"].corr(returns["IBM"])

np.float64(0.49976361144151166)

In [100]:
returns["MSFT"].cov(returns["IBM"])

np.float64(8.870655479703549e-05)

另一方面，DataFrame的corr和cov方法分别返回一个完整的相关矩阵或协方差矩阵作为DataFrame：

In [101]:
returns.corr()

,AAPL,GOOG,IBM,MSFT
AAPL,1.000000,0.407919,0.386817,0.389695
GOOG,0.407919,1.000000,0.405099,0.465919
IBM,0.386817,0.405099,1.000000,0.499764
MSFT,0.389695,0.465919,0.499764,1.000000


In [102]:
returns.cov()

,AAPL,GOOG,IBM,MSFT
AAPL,0.000277,0.000107,0.000078,0.000095
GOOG,0.000107,0.000251,0.000078,0.000108
IBM,0.000078,0.000078,0.000146,0.000089
MSFT,0.000095,0.000108,0.000089,0.000215


使用DataFrame的corrwith方法，你可以计算一个DataFrame的列或行与另一个Series或DataFrame之间的成对相关性。传递一个Series会返回一个Series，其中包含每个列计算出的相关性值：

In [103]:
returns.corrwith(returns["IBM"])

AAPL    0.386817
GOOG    0.405099
IBM     1.000000
MSFT    0.499764
dtype: float64

传递一个DataFrame计算匹配列名的相关性。这里，我计算百分比变化与成交量之间的相关性：

In [104]:
returns.corrwith(volume)

AAPL   -0.075565
GOOG   -0.007067
IBM    -0.204849
MSFT   -0.092950
dtype: float64

通过axis="columns"参数可以逐行处理。在所有情况下，计算相关性之前都会按标签对齐数据点。

### 唯一值、值计数和成员身份

另一类相关方法提取了一维序列中包含的值的信息。为了说明这些，请考虑以下示例：

In [105]:
obj = pd.Series(["c", "a", "d", "a", "a", "b", "b", "c", "c"])

第一个函数是unique，它给你一个Series中唯一值的数组：

In [106]:
uniques = obj.unique()
uniques

array(['c', 'a', 'd', 'b'], dtype=object)

唯一值不一定会按它们首次出现的顺序返回，也不一定按排序顺序返回，但如果需要的话可以在事后对它们进行排序（uniques.sort()）。相关地，value_counts计算一个包含值频率的Series：

In [107]:
obj.value_counts()

c    3
a    3
b    2
d    1
Name: count, dtype: int64

该系列按值降序排列以便于使用。value_counts也可以作为一个顶层pandas方法，与NumPy数组或其他Python序列一起使用：

In [108]:
pd.value_counts(obj.to_numpy(), sort=False)

/var/folders/c5/vh03t8zn4797kc18lgrjtcbr0000gn/T/ipykernel_2020/164454357.py:1: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  pd.value_counts(obj.to_numpy(), sort=False)


c    3
a    3
d    1
b    2
Name: count, dtype: int64

isin执行向量化的集合成员资格检查，在过滤数据集以获取Series或DataFrame中列的子集值时非常有用：

In [109]:
obj

0    c
1    a
2    d
3    a
4    a
5    b
6    b
7    c
8    c
dtype: object

In [110]:
mask = obj.isin(["b", "c"])
mask

0     True
1    False
2    False
3    False
4    False
5     True
6     True
7     True
8     True
dtype: bool

In [111]:
obj[mask]

0    c
5    b
6    b
7    c
8    c
dtype: object

与isin相关的是Index.get_indexer方法，它从一个可能包含非唯一值的数组中返回一个索引数组到另一个包含唯一值的数组中：

In [112]:
to_match = pd.Series(["c", "a", "b", "b", "c", "a"])
unique_vals = pd.Series(["c", "b", "a"])

indices = pd.Index(unique_vals).get_indexer(to_match)
indices

array([0, 2, 1, 1, 0, 2])

**唯一值计数、集合成员资格方法:**

| 方法 | 描述 |
|-----|------|
| isin | 计算一个布尔数组，指示每个Series或DataFrame值是否包含在传递的值序列中 |
| get_indexer | 计算数组中每个值对应的整数索引，并将其存储到另一个包含不同值的数组中；有助于数据对齐和连接型操作 |
| unique | 计算序列中唯一值的数组，按观察到的顺序返回 |
| value_counts | 返回一个Series对象，其索引包含唯一值，值为频率，按降序排列的计数 |

在某些情况下，您可能希望在DataFrame中的多个相关列上计算直方图。这里有一个例子：

In [113]:
data = pd.DataFrame({"Qu1": [1, 3, 4, 3, 4],
                     "Qu2": [2, 3, 1, 2, 3],
                     "Qu3": [1, 5, 2, 4, 4]})
data

,Qu1,Qu2,Qu3
0,1,2,1
1,3,3,5
2,4,1,2
3,3,2,4
4,4,3,4


我们可以计算单个列的值计数，如下所示：

In [114]:
data["Qu1"].value_counts().sort_index()

Qu1
1    1
3    2
4    2
Name: count, dtype: int64

要计算所有列的值计数，请将pandas.value_counts传递给DataFrame的apply方法：

In [115]:
result = data.apply(pd.value_counts).fillna(0)
result

/var/folders/c5/vh03t8zn4797kc18lgrjtcbr0000gn/T/ipykernel_2020/1382616601.py:1: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  result = data.apply(pd.value_counts).fillna(0)


,Qu1,Qu2,Qu3
1,1.0,1.0,1.0
2,0.0,2.0,1.0
3,2.0,2.0,0.0
4,2.0,0.0,2.0
5,0.0,0.0,1.0


这里，结果中的行标签是所有列中出现的不同值。这些值是每列中这些值的相应计数。

还有一个DataFrame.value_counts方法，但它计算计数时考虑将DataFrame的每一行作为一个元组来确定每个不同行的出现次数：

In [116]:
data = pd.DataFrame({"a": [1, 1, 1, 2, 2], "b": [0, 0, 1, 0, 0]})

data

,a,b
0,1,0
1,1,0
2,1,1
3,2,0
4,2,0


In [117]:
data.value_counts()

a  b
1  0    2
2  0    2
1  1    1
Name: count, dtype: int64

在这种情况下，结果具有一个索引，该索引将不同的行表示为一个层次结构索引，我们将在第8章“数据整理：连接、组合和重塑”中更详细地探讨这个话题。